# E-Commerce Order Analytics Pipeline

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import *

### Ingest & Validate

- Read the CSV file into a DataFrame with PySpark, enabling header detection
- Define an explicit schema for all columns upfront rather than inferring it
- Configure PySpark to handle malformed records gracefully by routing bad rows to a separate column instead of crashing the job

In [0]:
orders_schema = StructType([
    StructField('order_id', IntegerType(), True),
    StructField('customer_name', StringType(), True),
    StructField('customer_email', StringType(), True),
    StructField('product_category', StringType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('unit_price', DoubleType(), True),
    StructField('order_date', DateType(), True),
    StructField('status', StringType(), True),
    StructField('country', StringType(), True),
    StructField('_corrupt_record', StringType(), True)
])

In [0]:
orders_raw_df = spark.read.format('csv')\
                .option("header", True)\
                .option('mode', 'PERMISSIVE')\
                .option('columnNameOfCorruptRecord', '_corrupt_record')\
                .schema(orders_schema)\
                .load("/Volumes/workspace/learning_spark/spark/mini_projects_dataset/orders_raw_project_2.csv")
    
display(orders_raw_df)

### Audit Corrupt Records
- Count total records loaded
- Filter and view corrupt records
- Count corrupt records
- Count clean records
- Print audit summary

In [0]:
orders_raw_df.cache()

# .cache() not supported on Databricks Serverless
# corrupt_df count show 0 
# Visually verified 2 corrupt records via display()

In [0]:
total_count = orders_raw_df.count()

corrupt_df = orders_raw_df.filter(col('_corrupt_record').isNotNull())

corrupt_count = corrupt_df.count()

clean_count = orders_raw_df.filter(col('_corrupt_record').isNull()).count()

print(f"Total Records Loaded  : {total_count}")
print(f"Corrupt Records       : {corrupt_count}")
print(f"Clean Records         : {clean_count}")

display(corrupt_df)

### Clean the Data

- Drop the internal metadata columns that aren't needed for analysis (customer_email and the malformed record column)
- Rename unit_price to price_per_unit and quantity to order_quantity for clarity
- Cast price_per_unit and order_quantity to their correct numeric types 
- Remove duplicate orders, keeping only one record per order_id

In [0]:
cleaned_orders_df  = orders_raw_df\
                    .drop('customer_email', '_corrupt_record')\
                    .withColumnRenamed('unit_price', 'price_per_unit')\
                    .withColumnRenamed('quantity', 'order_quantity')\
                    .dropDuplicates(['order_id'])


display(cleaned_orders_df)

# Cast price_per_unit and order_quantity to their correct numeric types - Since the typed schema is defined upfront in Task 1, quantity is already IntegerType and unit_price is already DoubleType when the DataFrame is created

### Enrich the Data

- Add a new column total_order_value which is order_quantity × price_per_unit
- Add a new column is_high_value which flags orders where total_order_value exceeds ₹5,000 as "YES", otherwise "NO"
- Select only the columns relevant for the final report: order_id, customer_name, product_category, order_quantity, price_per_unit, total_order_value, is_high_value, order_date, status, country

In [0]:
enriched_orders_df = cleaned_orders_df\
                    .withColumn(
                            'total_order_value', 
                            col('order_quantity') * col('price_per_unit')
                    )\
                    .withColumn(
                            'is_high_value', 
                            when(col('total_order_value') > 5000, 'YES')
                            .otherwise('NO')
                    )

display(enriched_orders_df)

# Skipping the SELECT, since all required columns are already present and no extras need removing at this stage

### Analyze & Export

- Filter to only DELIVERED orders
- Sort the results by total_order_value in descending order
- Show only the Top 20 highest-value delivered orders
- As a bonus, show a separate filtered view of only CANCELLED orders from India

In [0]:
top_20_delivered_orders_df = enriched_orders_df\
                            .filter(upper(col('status')) == 'DELIVERED')\
                            .sort(col('total_order_value').desc())\
                            .limit(20)

display(top_20_delivered_orders_df)

In [0]:
india_cancelled_orders_df = enriched_orders_df\
                            .filter(
                                (col('country') == 'India') &
                                ((upper(col('status'))) == 'CANCELLED')
                            )

display(india_cancelled_orders_df)